In [113]:
import torch


model = torch.hub.load("facebookresearch/swag", model="vit_b16_in1k")

# we also convert the model to eval mode
model.eval()

resolution = 384

Using cache found in /Users/vaibhav/.cache/torch/hub/facebookresearch_swag_main


In [114]:
!curl https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json -O in_cls_idx.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 35363  100 35363    0     0  23998      0  0:00:01  0:00:01 --:--:-- 23991
curl: (6) Could not resolve host: in_cls_idx.json


In [115]:
import json


with open("imagenet_class_index.json", "r") as f:
    imagenet_id_to_name = {int(cls_id): name for cls_id, (label, name) in json.load(f).items()}

# let's preview the first few class names 
dict(sorted(imagenet_id_to_name.items()))

{0: 'tench',
 1: 'goldfish',
 2: 'great_white_shark',
 3: 'tiger_shark',
 4: 'hammerhead',
 5: 'electric_ray',
 6: 'stingray',
 7: 'cock',
 8: 'hen',
 9: 'ostrich',
 10: 'brambling',
 11: 'goldfinch',
 12: 'house_finch',
 13: 'junco',
 14: 'indigo_bunting',
 15: 'robin',
 16: 'bulbul',
 17: 'jay',
 18: 'magpie',
 19: 'chickadee',
 20: 'water_ouzel',
 21: 'kite',
 22: 'bald_eagle',
 23: 'vulture',
 24: 'great_grey_owl',
 25: 'European_fire_salamander',
 26: 'common_newt',
 27: 'eft',
 28: 'spotted_salamander',
 29: 'axolotl',
 30: 'bullfrog',
 31: 'tree_frog',
 32: 'tailed_frog',
 33: 'loggerhead',
 34: 'leatherback_turtle',
 35: 'mud_turtle',
 36: 'terrapin',
 37: 'box_turtle',
 38: 'banded_gecko',
 39: 'common_iguana',
 40: 'American_chameleon',
 41: 'whiptail',
 42: 'agama',
 43: 'frilled_lizard',
 44: 'alligator_lizard',
 45: 'Gila_monster',
 46: 'green_lizard',
 47: 'African_chameleon',
 48: 'Komodo_dragon',
 49: 'African_crocodile',
 50: 'American_alligator',
 51: 'triceratops',
 

In [116]:
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt


def load_image(image_path):
    return Image.open(image_path).convert("RGB")

def visualize_image(image):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

def transform_image(image, resolution):
    transform = transforms.Compose([
        transforms.Resize(
            resolution,
            interpolation=transforms.InterpolationMode.BICUBIC,
        ),
        transforms.CenterCrop(resolution),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ])
    image = transform(image)
    # we also add a batch dimension to the image since that is what the model expects
    image = image[None, :]
    return image

In [120]:
def visualize_and_predict(model, resolution, image_path):
    image = load_image(image_path)
    #visualize_image(image)
    print(image.size)  # PIL images use .size (width, height)
    image = transform_image(image, resolution)
    print(image.shape)
    # we do not need to track gradients for inference
    with torch.no_grad():
        _, preds = model(image).topk(5)
    # To print intermediate outputs of a model, you can register forward hooks on the layers of interest.
    # Example: Print the output of the first transformer block (or any named submodule)
    # This assumes the model is a torchvision/huggingface ViT or similar with named submodules.
    # You may need to adjust the submodule name depending on your model's architecture.

    # Define a hook function
    def print_intermediate_output(module, input, output):
        print(f"Intermediate output from {module.__class__.__name__}:")
        print("Shape:", output.shape)

    # Register the hook on a specific submodule (e.g., the first block)
    # For torchvision.models.vit_b_16, the first block is model.encoder.layers[0]
    # For timm/huggingface, it may be model.blocks[0] or similar.
    # Adjust as needed for your model:
    # Example for torchvision ViT:
    # handle = model.encoder.layers[0].register_forward_hook(print_intermediate_output)
    # Example for timm/huggingface ViT:
    # handle = model.blocks[0].register_forward_hook(print_intermediate_output)

    # For demonstration, try to register on a common ViT block name:
    
    handle = model.encoder.layers[0].ln_1.register_forward_hook(print_intermediate_output)
    
    # convert preds to a Python list and remove the batch dimension
    preds = preds.tolist()[0]
    print([imagenet_id_to_name[cls_id] for cls_id in preds])

In [121]:
visualize_and_predict(model, resolution, "dog.jpg")

(640, 480)
torch.Size([1, 3, 384, 384])
Intermediate output from LayerNorm:
Shape: torch.Size([577, 1, 768])
['dingo', 'alp', 'kelpie', 'Ibizan_hound', 'dhole']


In [127]:
# The best way to visualize a PyTorch model is to use torchsummary for a summary or torchviz for a graphical representation.
# Example 1: Print a summary of the model (shows layers, output shapes, params)
from torchsummary import summary
# The model expects input of shape (batch_size, 3, resolution, resolution)
# Use batch size 1 for summary
# torchsummary.summary may not work with all models (especially some HuggingFace/ViT models).
# Instead, print the model architecture and count parameters manually as a fallback.
print(model)
total_params = sum(p.numel() for p in model.encoder.layers.layer_0.parameters())
print(f"Total parameters: {total_params:,}")


ViTB16(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0, inplace=False)
    (layers): Sequential(
      (layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (linear_1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (dropout_1): Dropout(p=0, inplace=False)
          (linear_2): Linear(in_features=3072, out_features=768, bias=True)
          (dropout_2): Dropout(p=0, inplace=False)
        )
      )
      (layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_atte